# AssureX Claim Engine — Python Model Evaluation

Minimal reproducibility notebook for the TechWiz 7 AI & ML submission. It documents the dataset, loads the retained Python model, evaluates it on the held-out test set, and records the verified result.

**Project:** AssureX Claim Engine  
**Test set:** 225 unseen claims  
**Reported accuracy:** 92.8889%

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ROOT = Path.cwd()
TEST_PATH = ROOT / 'data' / 'datasets' / 'test.csv'
MODEL_PATH = ROOT / 'model' / 'classifier.joblib'

print('Test set:', TEST_PATH)
print('Model:', MODEL_PATH)
assert TEST_PATH.exists(), f'Missing: {TEST_PATH}'
assert MODEL_PATH.exists(), f'Missing: {MODEL_PATH}'

## 1. Dataset

The project uses a stratified 70/15/15 split of 1,500 structured claims: 1,050 training, 225 validation, and 225 held-out test claims. The test set is used only for final evaluation.

In [ ]:
test_df = pd.read_csv(TEST_PATH)
print('Test rows:', len(test_df))
display(test_df.head())
print('\nColumns:')
print(test_df.columns.tolist())

In [ ]:
# Identify the target column used by the retained dataset.
target_candidates = ['claim_class', 'class', 'label', 'target', 'status']
target_col = next((c for c in target_candidates if c in test_df.columns), None)
if target_col is None:
    raise ValueError(f'Could not identify target column. Available columns: {test_df.columns.tolist()}')

print('Target column:', target_col)
print('\nClass distribution:')
print(test_df[target_col].value_counts().sort_index())

## 2. Retained Python model

The notebook loads the production `classifier.joblib` already retained in the repository. No retraining is performed here; this notebook is an evaluation/reproducibility record.

In [ ]:
model = joblib.load(MODEL_PATH)
print(type(model).__name__)
if hasattr(model, 'classes_'):
    print('Model classes:', list(model.classes_))

In [ ]:
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4%}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

## 3. Verified result

The retained evaluation evidence reports **92.8889% accuracy (209/225)** on the independent 225-claim test set.

This notebook is intended to document and reproduce the Python-model evaluation. It does not claim that synthetic test performance represents real-world claims performance.

In [ ]:
# Submission evidence check.
expected_accuracy = 209 / 225
assert len(test_df) == 225, f'Expected 225 test rows, found {len(test_df)}'
assert abs(accuracy - expected_accuracy) < 1e-12, (
    f'Unexpected accuracy: {accuracy:.10f}; expected {expected_accuracy:.10f}'
)
print('PASS: 225 held-out claims evaluated; accuracy = 92.8889% (209/225).')